# Realtime Voice Interpreter — EN ↔ PT Translator

Build a low-latency, two-way **English ↔ Portuguese** voice interpreter on the **Realtime API** (`gpt-realtime-1.5`). The model listens to speech in one language and speaks the translation in the other, in near real time, over a persistent **WebSocket** connection.

> ## ⚠️ TODO — requirements before this runs
>
> This notebook **will not run as-is in a plain Jupyter cell**. It requires:
> - **Microphone + speaker access** (and a real audio I/O loop) — notebooks don't capture mic audio by default.
> - The **GA Realtime API** and the `gpt-realtime-1.5` model enabled on your account.
> - An async event loop for the WebSocket session (run the audio loop from a script, not a single cell).
>
> Treat the cells below as the **connection + session pattern**. Wire them into a small local app (or `python realtime_interpreter.py`) with a real audio capture/playback loop before expecting end-to-end translation. Execution is deferred — students need API keys, the GA Realtime model, and mic hardware.

## Setup

In [ ]:
import os
import getpass

def _set_env(var):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("OPENAI_API_KEY")

# Audio + websocket deps (install once):
# !pip install -U openai websockets sounddevice numpy

## 1. Session configuration

The Realtime session is configured with the model, the voice, the audio formats, and — crucially for an interpreter — **instructions that pin the translation behavior**.

In [ ]:
REALTIME_MODEL = "gpt-realtime-1.5"
VOICE = "marin"   # pick any available realtime voice

INTERPRETER_INSTRUCTIONS = """
You are a live bidirectional interpreter between English and Portuguese.
- If the speaker uses English, respond ONLY with the Portuguese translation (spoken).
- If the speaker uses Portuguese, respond ONLY with the English translation (spoken).
- Do not add commentary, greetings, or explanations. Translate faithfully, preserving tone.
- Keep latency low: begin speaking as soon as a complete phrase is available.
"""

session_config = {
    "model": REALTIME_MODEL,
    "voice": VOICE,
    "instructions": INTERPRETER_INSTRUCTIONS,
    "input_audio_format": "pcm16",
    "output_audio_format": "pcm16",
    "input_audio_transcription": {"model": "whisper-1"},  # see what was heard
    "turn_detection": {"type": "server_vad"},             # auto-detect speech turns
}

## 2. WebSocket connection pattern

Open a persistent WebSocket to the Realtime endpoint, send the session config, then pump microphone audio in and play translated audio out. The skeleton below shows the connection + event-handling shape. *(The exact endpoint/header names are [unverified] — confirm against the GA Realtime docs.)*

In [ ]:
import asyncio
import json
import base64
import websockets   # pip install websockets

REALTIME_URL = "wss://api.openai.com/v1/realtime"  # [unverified] confirm GA path

async def realtime_interpreter():
    headers = {
        "Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}",
        "OpenAI-Beta": "realtime=v1",   # [unverified] header for GA
    }
    async with websockets.connect(
        f"{REALTIME_URL}?model={REALTIME_MODEL}", extra_headers=headers
    ) as ws:
        # 1) Configure the session as an interpreter.
        await ws.send(json.dumps({"type": "session.update", "session": session_config}))

        # 2) Concurrently: send mic audio, receive translated audio.
        async def send_mic_audio():
            # TODO: replace with a real mic capture loop (sounddevice).
            # For each PCM16 chunk `chunk_bytes`:
            #   await ws.send(json.dumps({
            #       "type": "input_audio_buffer.append",
            #       "audio": base64.b64encode(chunk_bytes).decode(),
            #   }))
            await asyncio.sleep(0)  # placeholder

        async def receive_events():
            async for message in ws:
                event = json.loads(message)
                etype = event.get("type", "")
                if etype == "response.audio.delta":
                    audio = base64.b64decode(event["delta"])
                    # TODO: play `audio` (PCM16) on the speaker (sounddevice).
                elif etype == "response.audio_transcript.delta":
                    print(event.get("delta", ""), end="", flush=True)  # live transcript
                elif etype == "error":
                    print("Realtime error:", event)

        await asyncio.gather(send_mic_audio(), receive_events())

# To run from a script:  asyncio.run(realtime_interpreter())
print("Realtime interpreter session pattern defined. Wire in mic/speaker I/O to run end-to-end.")

## 3. Where to go from here

- Add a real mic capture loop (`sounddevice.InputStream`) feeding `input_audio_buffer.append`.
- Play `response.audio.delta` chunks through `sounddevice.OutputStream`.
- Use `server_vad` turn detection so the interpreter responds as soon as the speaker pauses.
- For lowest latency, keep `reasoning` light — translation is not a deep-reasoning task.

This is the same WebSocket transport mentioned in the Responses API intro (`00-introduction-openai-responses-api.ipynb`).